# Dados e primeira análise com LLM

## 📌 Contexto e Objetivos
Este notebook consolida a auditoria e o saneamento preliminar dos dados transacionais legados no contexto de **PLD/AML (Prevenção à Lavagem de Dinheiro)**.

O fluxo divide-se em duas partes:
1. **Parte 1:** Tratamento e higienização dos dados (remoção de duplicatas, limpeza de caracteres invisíveis e sinalização mandatória de campos com valores nulos).
2. **Parte 2:** Análise da janela temporal (espaço entre a primeira e a última data) para viabilizar análises futuras e contextualizar operações com inconsistências.

---  
## 1. Tratamento dos Dados e Sinalização de Valores Nulos

In [1]:
import os
import json
import unicodedata
import pandas as pd
from datetime import datetime

# 1. Carregamento dos dados brutos
caminho_dados = os.path.join("..", "dados", "dados_nivel_1.json")
with open(caminho_dados, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

taxa_cambio = raw_data.get("taxa_cambio_usd_brl", 5.4)
df = pd.DataFrame(raw_data["operacoes"])

# Função para remoção de caracteres invisíveis e de controle
def limpar_texto(valor):
    if isinstance(valor, str):
        return "".join(c for c in valor if unicodedata.category(c)[0] != "C").strip()
    return valor

# Higienização textual
df = df.map(limpar_texto)

# Deduplicação de registros por ID
df_tratado = df.drop_duplicates(subset=["id"], keep="first").copy()

# Sinalização de campos nulos
condicao_nulo = (
    df_tratado.isnull().any(axis=1) | 
    df_tratado.isin(["NULO", "NULL", "None", "nan"]).any(axis=1)
)
registros_nulos = df_tratado[condicao_nulo]

print("=== RELATÓRIO DE HIGIENIZAÇÃO E AUDITORIA DE DADOS ===")
print(f"Total de registros válidos pós-deduplicação: {len(df_tratado)}")
print(f"Total de operações com campos nulos identificadas: {len(registros_nulos)}\n")

for _, row in registros_nulos.iterrows():
    print("⚠️ ALERTA DE CAMPO NULO:")
    print(f"   • ID da Operação: {row['id']}")
    print(f"   • Cliente: {row['cliente_id']}")
    print(f"   • Campo com Nulo: 'data' -> {row['data']}")
    print(f"   • Canal / Tipo: {row['canal']} / {row['tipo']}")
    print(f"   • Valor: R$ {row['valor']:,.2f}")
    print(f"   • Observação do Legado: '{row['observacao']}'")

=== RELATÓRIO DE HIGIENIZAÇÃO E AUDITORIA DE DADOS ===
Total de registros válidos pós-deduplicação: 19
Total de operações com campos nulos identificadas: 1

⚠️ ALERTA DE CAMPO NULO:
   • ID da Operação: OP-0017
   • Cliente: CLI-A-5
   • Campo com Nulo: 'data' -> None
   • Canal / Tipo: especie / deposito
   • Valor: R$ 4,300.00
   • Observação do Legado: 'data nao capturada pelo sistema'


---  
## 2. Análise da Janela Temporal (Espaço entre Datas)

Identificamos os extremos cronológicos das transações válidas para estruturar a janela operacional e avaliar hipóteses de análise futura.

In [2]:
# Filtragem de registros com datas válidas
df_temporal = df_tratado[df_tratado["data"].notnull() & (df_tratado["data"] != "")].copy()
df_temporal["data_dt"] = pd.to_datetime(df_temporal["data"], format="%Y-%m-%d")

primeira_data = df_temporal["data_dt"].min()
ultima_data = df_temporal["data_dt"].max()
intervalo_dias = (ultima_data - primeira_data).days

op_primeira = df_temporal.loc[df_temporal["data_dt"].idxmin()]
op_ultima = df_temporal.loc[df_temporal["data_dt"].idxmax()]

print("=== ANÁLISE DO ESPAÇO TEMPORAL ENTRE AS DATAS ===")
print(f"• Primeira transação registrada: {primeira_data.strftime('%d/%m/%Y')} ({primeira_data.strftime('%d/%m')})")
print(f"  └─ Operação: {op_primeira['id']} | Cliente: {op_primeira['cliente_id']} | Valor: R$ {op_primeira['valor']:,.2f}")
print(f"• Última transação registrada:   {ultima_data.strftime('%d/%m/%Y')} ({ultima_data.strftime('%d/%m')})")
print(f"  └─ Operação: {op_ultima['id']} | Cliente: {op_ultima['cliente_id']} | Valor: R$ {op_ultima['valor']:,.2f}")
print(f"• Espaço temporal total:         {intervalo_dias} dias (de {primeira_data.strftime('%d/%m')} a {ultima_data.strftime('%d/%m')})")

=== ANÁLISE DO ESPAÇO TEMPORAL ENTRE AS DATAS ===
• Primeira transação registrada: 03/03/2026 (03/03)
  └─ Operação: OP-0010 | Cliente: CLI-A-4 | Valor: R$ 3,800.00
• Última transação registrada:   28/03/2026 (28/03)
  └─ Operação: OP-0019 | Cliente: CLI-A-6 | Valor: R$ 1,400.00
• Espaço temporal total:         25 dias (de 03/03 a 28/03)
